# 🛰️ Starlink Constellation Collision Risk Analysis
### Optimized End-to-End Pipeline
**Author:** Writick Parui | M.E. CSE, TIET Patiala

**Sections:**
1. Setup & Imports
2. TLE Fetch & Parse
3. Orbital Propagation (SGP4)
4. Altitude Density Analysis
5. KD-Tree Proximity & Collision Detection
6. Satellite Brightness Modeling
7. SatNOGS Observation Fetch & Calibration
8. ML Brightness Predictor (Gradient Boosting)
9. Collision Risk Scoring & ML Classifier
10. Multi-Satellite Dataset Generation (100 sats)
11. Plotly Interactive Dashboard (HTML export)
12. Power BI Export Pipeline (6 structured CSVs)
13. Summary & File Export


In [ ]:
!pip install -q sgp4 skyfield pandas numpy matplotlib scipy \
             requests tqdm scikit-learn plotly python-dateutil joblib seaborn

In [ ]:
# ============================================================
# STARLINK CONSTELLATION COLLISION RISK ANALYSIS
# Optimized End-to-End Pipeline
# Author: Writick Parui | M.E. CSE, TIET Patiala
#
# Sections:
#   1. Setup & Imports
#   2. TLE Fetch & Parse
#   3. Orbital Propagation (SGP4)
#   4. Altitude Density Analysis
#   5. KD-Tree Proximity & Collision Detection
#   6. Satellite Brightness Modeling
#   7. SatNOGS Observation Fetch & Calibration
#   8. ML Brightness Predictor (Gradient Boosting)
#   9. Collision Risk Scoring & ML Classifier
#  10. Multi-Satellite Dataset Generation (100 sats)
#  11. Plotly Interactive Dashboard (HTML export)
#  12. Power BI Export Pipeline (6 structured CSVs)
#  13. Summary & File Export
# ============================================================

## Section 1 — Setup & Imports

In [ ]:
# ============================================================
# SECTION 1 — SETUP & IMPORTS
# ============================================================

# Install all dependencies in one shot
# !pip install -q sgp4 skyfield pandas numpy matplotlib scipy \
#              requests tqdm scikit-learn plotly python-dateutil joblib seaborn

import os, json, math, time, requests, warnings
from datetime import datetime, timedelta, timezone
from dateutil import parser as dateparser

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # headless backend — safe for Colab and local
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial import cKDTree
from tqdm import tqdm

from sgp4.api import Satrec, jday
from skyfield.api import Loader, EarthSatellite, wgs84

import joblib
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve, auc,
                             mean_squared_error, r2_score)
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# ── Global config ──────────────────────────────────────────
WORKDIR   = '/content/starlink_env_impact'
DATA_DIR  = os.path.join(WORKDIR, 'data')
EXPORT    = os.path.join(WORKDIR, 'export')
for d in [WORKDIR, DATA_DIR, EXPORT]:
    os.makedirs(d, exist_ok=True)

R_EARTH   = 6371.0        # km
N_SATS    = 100           # synthetic constellation size
N_FRAMES  = 300           # propagation timesteps
ORBIT_ALT = 550           # km (Starlink shell)
THRESH_KM = 5.0           # close-approach threshold
SEARCH_R  = 100.0         # KD-Tree candidate search radius
np.random.seed(42)

print("✔  Workspace:", WORKDIR)
print("✔  All imports loaded.")

## Section 2 — TLE Fetch & Parse

In [ ]:
# ============================================================
# SECTION 2 — TLE FETCH & PARSE
# ============================================================

CELESTRAK_URLS = [
    'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle',
    'https://celestrak.com/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle',
    'https://celestrak.com/NORAD/elements/starlink.txt',
]

def fetch_tle(out_path: str = os.path.join(DATA_DIR, 'starlink_latest.txt')) -> str:
    """
    Download the latest Starlink TLE file from CelesTrak.
    Tries multiple fallback URLs. Returns the path to the saved file.
    Raises RuntimeError if all sources fail.
    """
    for url in CELESTRAK_URLS:
        try:
            print(f"  Fetching TLEs from {url}...")
            r = requests.get(url, timeout=30)
            if r.status_code == 200 and '1 ' in r.text and '2 ' in r.text:
                with open(out_path, 'w') as f:
                    f.write(r.text)
                n_lines = r.text.strip().count('\n') + 1
                print(f"  ✔  Saved {n_lines} lines → {out_path}")
                return out_path
            print(f"  ✗  Bad response from {url} (status={r.status_code})")
        except Exception as e:
            print(f"  ✗  Error: {e}")
    raise RuntimeError("All TLE sources failed. Check network access.")


def parse_tle_file(path: str) -> list:
    """
    Parse a standard TLE text file.
    Handles both 2-line (no name) and 3-line (name + 2 lines) formats.
    Returns list of (name, line1, line2) tuples.
    """
    lines = [ln.strip() for ln in open(path) if ln.strip()]
    tles, i = [], 0
    while i < len(lines) - 1:
        # 3-line format: name, line1, line2
        if (i + 2 < len(lines)
                and lines[i + 1].startswith('1 ')
                and lines[i + 2].startswith('2 ')):
            tles.append((lines[i], lines[i+1], lines[i+2]))
            i += 3
        # 2-line format: line1, line2 (no name)
        elif lines[i].startswith('1 ') and lines[i+1].startswith('2 '):
            tles.append((f'SAT_{i}', lines[i], lines[i+1]))
            i += 2
        else:
            i += 1
    print(f"  ✔  Parsed {len(tles)} TLE records")
    return tles


# ── Run fetch + parse ──────────────────────────────────────
tle_path = fetch_tle()
tles     = parse_tle_file(tle_path)

## Section 3 — Orbital Propagation (SGP4)

In [ ]:
# ============================================================
# SECTION 3 — ORBITAL PROPAGATION (SGP4)
# ============================================================

def tle_to_satrec(tle_list: list) -> list:
    """Convert (name, l1, l2) tuples to (name, Satrec) pairs."""
    sats = []
    for name, l1, l2 in tle_list:
        try:
            sats.append((name, Satrec.twoline2rv(l1, l2)))
        except Exception as e:
            print(f"  ⚠  Bad TLE for {name}: {e}")
    return sats


def propagate(satrec_objs: list,
              start_dt: datetime,
              end_dt: datetime,
              step_seconds: int = 300,
              max_sats: int = 500) -> pd.DataFrame:
    """
    Propagate satellites over a time window using SGP4.

    Args:
        satrec_objs : list of (name, Satrec)
        start_dt    : window start (UTC datetime)
        end_dt      : window end   (UTC datetime)
        step_seconds: propagation step
        max_sats    : cap number of sats to avoid memory issues

    Returns:
        DataFrame with columns [sat, timestamp, x, y, z, vx, vy, vz]
    """
    times = []
    dt = start_dt
    while dt <= end_dt:
        times.append(dt)
        dt += timedelta(seconds=step_seconds)

    rows = []
    for name, sat in tqdm(satrec_objs[:max_sats], desc='Propagating'):
        for dt in times:
            jd, fr = jday(dt.year, dt.month, dt.day,
                          dt.hour, dt.minute,
                          dt.second + dt.microsecond * 1e-6)
            e, r, v = sat.sgp4(jd, fr)
            if e == 0 and None not in r:
                rows.append({
                    'sat': name, 'timestamp': dt,
                    'x': r[0], 'y': r[1], 'z': r[2],
                    'vx': v[0], 'vy': v[1], 'vz': v[2],
                })
    df = pd.DataFrame(rows)
    if not df.empty:
        df['alt_km'] = np.sqrt(df.x**2 + df.y**2 + df.z**2) - R_EARTH
    print(f"  ✔  Propagated {len(df):,} position records")
    return df


# ── Run propagation for a 6-hour window ───────────────────
start_dt = datetime.utcnow().replace(minute=0, second=0, microsecond=0)
end_dt   = start_dt + timedelta(hours=6)
satrec_objs = tle_to_satrec(tles)
df_pos = propagate(satrec_objs, start_dt, end_dt, step_seconds=300, max_sats=500)

## Section 4 — Altitude Density Analysis

In [ ]:
# ============================================================
# SECTION 4 — ALTITUDE DENSITY ANALYSIS
# ============================================================

def compute_altitude_density(df_pos: pd.DataFrame,
                              bins_km: np.ndarray = np.arange(200, 1001, 50)) -> dict:
    """
    Compute histogram of satellite counts per altitude band.
    Returns dict with bin edges and counts.
    """
    if df_pos.empty:
        return {}
    hist, edges = np.histogram(df_pos['alt_km'].dropna(), bins=bins_km)
    return {'edges': edges.tolist(), 'counts': hist.tolist()}


def plot_altitude_density(dens: dict, save_path: str = None):
    """Bar chart of satellite counts per altitude band."""
    if not dens:
        print("  ⚠  No density data to plot.")
        return
    centers = [(dens['edges'][i] + dens['edges'][i+1]) / 2
               for i in range(len(dens['counts']))]
    widths  = np.diff(dens['edges'])
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(centers, dens['counts'], width=widths, align='center',
           edgecolor='k', color='steelblue', alpha=0.85)
    ax.set_xlabel('Altitude (km)')
    ax.set_ylabel('Position samples')
    ax.set_title('Starlink Altitude Distribution (Sampled Positions)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  ✔  Saved → {save_path}")
    plt.show()


dens = compute_altitude_density(df_pos)
plot_altitude_density(dens, save_path=os.path.join(EXPORT, 'altitude_density.png'))

## Section 5 — KD-Tree Proximity & Collision Detection

In [ ]:
# ============================================================
# SECTION 5 — KD-TREE PROXIMITY & COLLISION DETECTION
# ============================================================

def compute_close_pairs_kdtree(df_pos: pd.DataFrame,
                                threshold_km: float = THRESH_KM,
                                sample_frac: float = 0.3) -> int:
    """
    Use scipy.spatial.cKDTree to count satellite pairs within threshold_km.

    KD-Tree reduces complexity from O(n²) to O(n log n).
    A random sample is taken if dataset > 2000 rows to keep it fast.

    Returns:
        Number of close-approach pairs detected.
    """
    df_s = df_pos.sample(frac=min(1.0, sample_frac)) if len(df_pos) > 2000 else df_pos
    coords = df_s[['x', 'y', 'z']].dropna().values
    if coords.shape[0] < 2:
        return 0
    tree  = cKDTree(coords)
    pairs = tree.query_pairs(r=threshold_km)
    return len(pairs)


def build_collision_features(df_pos: pd.DataFrame,
                              search_radius: float = SEARCH_R,
                              threshold_km: float = THRESH_KM) -> pd.DataFrame:
    """
    Build per-pair feature vectors for ML collision prediction.

    For each timestep snapshot, KD-Tree finds candidate pairs within
    search_radius. For each candidate pair, features are extracted:
      - distance_km, relative speed, range_rate,
        hist_close_count (historical encounters)

    Returns:
        DataFrame of (satA, satB, frame, distance_km, rel_speed,
                      range_rate, hist_close_count, label)
    """
    close_history = {}  # (satA, satB) → count
    rows = []
    frames = sorted(df_pos['timestamp'].unique())

    for frame_ts in tqdm(frames, desc='Building collision features'):
        snap = df_pos[df_pos['timestamp'] == frame_ts].dropna(
            subset=['x', 'y', 'z'])
        if len(snap) < 2:
            continue

        coords = snap[['x', 'y', 'z']].values
        names  = snap['sat'].values
        vels   = snap[['vx', 'vy', 'vz']].values \
                 if all(c in snap.columns for c in ['vx', 'vy', 'vz']) \
                 else np.zeros_like(coords)

        tree = cKDTree(coords)
        idxs = tree.query_pairs(r=search_radius)

        for i, j in idxs:
            d    = np.linalg.norm(coords[i] - coords[j])
            dv   = vels[i] - vels[j]
            dr   = coords[i] - coords[j]
            rel_speed  = np.linalg.norm(dv)
            range_rate = float(np.dot(dv, dr) / (d + 1e-9))
            key = tuple(sorted((names[i], names[j])))
            close_history[key] = close_history.get(key, 0) + (1 if d < threshold_km else 0)
            rows.append({
                'frame':           frame_ts,
                'satA':            names[i],
                'satB':            names[j],
                'distance_km':     d,
                'rel_speed':       rel_speed,
                'range_rate':      range_rate,
                'hist_close_count': close_history.get(key, 0),
                'label':           int(d < threshold_km),
            })

    df_feat = pd.DataFrame(rows)
    print(f"  ✔  Built {len(df_feat):,} pair-feature records "
          f"| positives: {df_feat['label'].sum():,}")
    return df_feat


n_pairs = compute_close_pairs_kdtree(df_pos)
print(f"  ✔  Close approach pairs (sampled): {n_pairs}")

df_features = build_collision_features(df_pos, search_radius=SEARCH_R, threshold_km=THRESH_KM)
df_features.to_csv(os.path.join(EXPORT, 'collision_features.csv'), index=False)

## Section 6 — Satellite Brightness Modeling

In [ ]:
# ============================================================
# SECTION 6 — SATELLITE BRIGHTNESS MODELING
# ============================================================

OBSERVATORIES = [
    {"name": "Mauna Kea",  "lat":  19.8206, "lon": -155.4681, "elev_m": 4205},
    {"name": "Paranal",    "lat": -24.6252, "lon":  -70.4025, "elev_m": 2635},
    {"name": "Kitt Peak",  "lat":  31.9583, "lon": -111.5967, "elev_m": 2096},
    {"name": "Greenwich",  "lat":  51.4769, "lon":   -0.0005, "elev_m":   46},
    {"name": "Sydney",     "lat": -33.8688, "lon":  151.2093, "elev_m":   58},
]


def simple_mag_model(elevation_deg: float,
                     phase_deg: float = 40.0,
                     base_mag: float = 5.5) -> float:
    """
    Physically-inspired magnitude estimator.
      m = m0 - 2.5 * log10(cos(zenith_angle) + ε) + k * phase_deg
    where zenith_angle = 90 - elevation_deg.
    """
    feat_elev = -2.5 * math.log10(
        max(math.cos(math.radians(90 - elevation_deg)), 1e-6))
    return base_mag + feat_elev + 0.02 * phase_deg


def compute_passes_skyfield(tle_path: str,
                             sites: list = None,
                             start_dt: datetime = None,
                             days: int = 1,
                             step_min: int = 1,
                             min_elevation: float = 15.0,
                             max_sats: int = 50) -> dict:
    """
    Compute satellite passes over observatories using Skyfield.

    For each (satellite, observatory) pair, finds time windows where
    satellite elevation exceeds min_elevation degrees.

    Returns:
        dict keyed by observatory name → list of pass dicts.
    """
    if sites is None:
        sites = OBSERVATORIES
    if start_dt is None:
        start_dt = datetime.utcnow().replace(
            minute=0, second=0, microsecond=0)

    load = Loader(os.path.join(WORKDIR, 'skyfield_data'))
    ts   = load.timescale()

    tles = parse_tle_file(tle_path)[:max_sats]

    # Build time array
    end_dt  = start_dt + timedelta(days=days)
    n_steps = int((end_dt - start_dt).total_seconds() // (step_min * 60)) + 1
    t_arr   = ts.utc(start_dt.year, start_dt.month, start_dt.day,
                     start_dt.hour,
                     [start_dt.minute + i * step_min for i in range(n_steps)])

    passes = {s['name']: [] for s in sites}

    for name, l1, l2 in tqdm(tles, desc='Computing passes'):
        try:
            sat = EarthSatellite(l1, l2, name, ts)
        except Exception:
            continue

        for site in sites:
            obs = wgs84.latlon(site['lat'], site['lon'],
                               elevation_m=site['elev_m'])
            diff = sat - obs
            alt, az, dist = diff.at(t_arr).altaz()
            el_arr = alt.degrees

            # Detect pass segments (elevation above threshold)
            above = el_arr > min_elevation
            in_pass = False
            pass_buf = []

            for idx, is_above in enumerate(above):
                if is_above:
                    pass_buf.append(idx)
                    in_pass = True
                elif in_pass:
                    if pass_buf:
                        max_el_idx = pass_buf[int(np.argmax(el_arr[pass_buf]))]
                        passes[site['name']].append({
                            'sat':           name,
                            'start_idx':     pass_buf[0],
                            'end_idx':       pass_buf[-1],
                            'max_elev':      float(el_arr[max_el_idx]),
                            'predicted_mag': simple_mag_model(
                                float(el_arr[max_el_idx])),
                        })
                    pass_buf = []
                    in_pass  = False

    total = sum(len(v) for v in passes.values())
    print(f"  ✔  Found {total} passes across {len(sites)} observatories")
    return passes, t_arr


passes, sf_times = compute_passes_skyfield(
    tle_path, start_dt=start_dt, days=1, max_sats=50)

## Section 7 — SatNOGS Observation Fetch & Calibration

In [ ]:
# ============================================================
# SECTION 7 — SATNOGS FETCH & CALIBRATION
# ============================================================

SATNOGS_DB  = "https://db.satnogs.org/api/satellites/"
SATNOGS_NET = "https://network.satnogs.org/api/observations/"


def fetch_satnogs_starlink(limit_pages: int = 5) -> list:
    """
    Query SatNOGS DB for Starlink satellite entries.
    Returns list of satellite dicts with satnogs_id and norad_cat_id.
    """
    results = []
    for page in range(1, limit_pages + 1):
        try:
            r = requests.get(SATNOGS_DB,
                             params={'page': page, 'format': 'json'},
                             timeout=20)
            r.raise_for_status()
            batch = r.json()
            if not batch:
                break
            for s in batch:
                if 'starlink' in (s.get('name') or '').lower():
                    results.append(s)
            time.sleep(0.3)
        except Exception as e:
            print(f"  ⚠  SatNOGS page {page} error: {e}")
            break
    print(f"  ✔  Found {len(results)} Starlink entries in SatNOGS DB")
    return results


def fetch_satnogs_observations(norad_id: int,
                                page_size: int = 100,
                                max_pages: int = 3) -> list:
    """
    Fetch recent observations for a satellite from SatNOGS Network.
    Returns list of observation dicts.
    """
    obs = []
    for page in range(1, max_pages + 1):
        try:
            r = requests.get(SATNOGS_NET, params={
                'satellite__norad_cat_id': norad_id,
                'page_size': page_size,
                'ordering': '-start',
                'page': page,
            }, timeout=20)
            r.raise_for_status()
            batch = r.json()
            if not batch:
                break
            obs.extend(batch if isinstance(batch, list)
                       else batch.get('results', []))
            time.sleep(0.3)
        except Exception as e:
            print(f"  ⚠  SatNOGS obs error (NORAD {norad_id}): {e}")
            break
    return obs


def build_calibration_dataset(passes: dict, sf_times,
                               satnogs_entries: list) -> pd.DataFrame:
    """
    Match SatNOGS observations to predicted passes by time window.
    Builds a calibration DataFrame with columns:
      feat_elev, phase_deg, predicted_mag, mag_obs
    Falls back to synthetic data if matches < 10.
    """
    # Build predicted-pass lookup keyed by satellite name
    pred_list = []
    for site, p_list in passes.items():
        for p in p_list:
            si = p.get('start_idx')
            ei = p.get('end_idx')
            try:
                s_dt = sf_times[si].utc_datetime() if si is not None else None
                e_dt = sf_times[ei].utc_datetime() if ei is not None else None
            except Exception:
                s_dt = e_dt = None
            pred_list.append({
                'site':          site,
                'sat':           p['sat'],
                'start_dt':      s_dt,
                'end_dt':        e_dt,
                'max_elev':      p['max_elev'],
                'predicted_mag': p['predicted_mag'],
            })

    rows = []
    for entry in satnogs_entries[:30]:
        norad = entry.get('norad_cat_id')
        if norad is None:
            continue
        obs_list = fetch_satnogs_observations(norad, max_pages=2)
        for obs in obs_list:
            if obs.get('vetted_status') != 'good':
                continue
            obs_start = dateparser.parse(obs.get('start', '')) if obs.get('start') else None
            if obs_start is None:
                continue
            # Match predicted pass within ±15 minutes
            for pred in pred_list:
                if pred['start_dt'] is None:
                    continue
                s_dt = pred['start_dt']
                if abs((obs_start - s_dt).total_seconds()) < 900:
                    elev = pred['max_elev']
                    feat = -2.5 * math.log10(
                        max(math.cos(math.radians(90 - elev)), 1e-6))
                    rows.append({
                        'feat_elev':     feat,
                        'phase_deg':     40.0,
                        'predicted_mag': pred['predicted_mag'],
                        'mag_obs':       float(obs.get('magnitude', np.nan)),
                    })

    df_cal = pd.DataFrame(rows).dropna()

    # Fallback: synthetic calibration if real matches < 10
    if len(df_cal) < 10:
        print("  ⚠  Real calibration matches < 10; using synthetic data")
        pred_mag = np.linspace(4, 8, 50)
        obs_mag  = pred_mag + np.random.normal(0.3, 0.25, 50)
        feat_elev = np.random.uniform(-0.5, 1.5, 50)
        df_cal = pd.DataFrame({
            'feat_elev':     feat_elev,
            'phase_deg':     np.full(50, 40.0),
            'predicted_mag': pred_mag,
            'mag_obs':       obs_mag,
        })

    print(f"  ✔  Calibration dataset: {len(df_cal)} rows")
    return df_cal


satnogs_entries = fetch_satnogs_starlink(limit_pages=3)
df_cal = build_calibration_dataset(passes, sf_times, satnogs_entries)
df_cal.to_csv(os.path.join(EXPORT, 'calibration_data.csv'), index=False)

## Section 8 — ML Brightness Predictor (Gradient Boosting)

In [ ]:
# ============================================================
# SECTION 8 — ML BRIGHTNESS PREDICTOR (GRADIENT BOOSTING)
# ============================================================

def train_brightness_model(df_cal: pd.DataFrame) -> dict:
    """
    Train a Gradient Boosting Regressor to predict calibrated
    apparent magnitude from:
      - feat_elev  : elevation-based photometric feature
      - phase_deg  : Sun–satellite–observer phase angle
      - predicted_mag: raw model prediction (optional)

    Returns dict with model, feature list, and eval metrics.
    """
    feature_candidates = ['feat_elev', 'phase_deg', 'predicted_mag']
    features = [f for f in feature_candidates if f in df_cal.columns]
    X = df_cal[features].values
    y = df_cal['mag_obs'].values

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42)

    model = GradientBoostingRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=42)
    model.fit(X_tr, y_tr)

    preds = model.predict(X_te)
    rmse  = np.sqrt(mean_squared_error(y_te, preds))
    r2    = r2_score(y_te, preds)
    print(f"  Brightness model — RMSE: {rmse:.3f} mag | R²: {r2:.3f}")

    # Save model
    mobj = {'model': model, 'features': features}
    joblib.dump(mobj, os.path.join(WORKDIR, 'brightness_gb_model.joblib'))
    return mobj


def apply_brightness_model(passes: dict, mobj: dict) -> pd.DataFrame:
    """
    Apply calibrated brightness model to all predicted satellite passes.
    Returns DataFrame with original + calibrated magnitude columns.
    """
    model    = mobj['model']
    features = mobj['features']
    rows = []
    for site, p_list in passes.items():
        for p in p_list:
            elev     = p.get('max_elev') or 30.0
            feat_elev = -2.5 * math.log10(
                max(math.cos(math.radians(90 - elev)), 1e-6))
            row = {
                'site':          site,
                'sat':           p['sat'],
                'max_elev':      elev,
                'predicted_mag': p.get('predicted_mag'),
            }
            data = {}
            if 'feat_elev'      in features: data['feat_elev']      = feat_elev
            if 'phase_deg'      in features: data['phase_deg']      = 40.0
            if 'predicted_mag'  in features: data['predicted_mag']  = p.get('predicted_mag', 6.0)
            if data:
                X_row = np.array([[data[f] for f in features]])
                row['calibrated_mag'] = float(model.predict(X_row)[0])
            rows.append(row)
    return pd.DataFrame(rows)


mobj = train_brightness_model(df_cal)
df_passes = apply_brightness_model(passes, mobj)
df_passes.to_csv(os.path.join(EXPORT, 'passes_calibrated.csv'), index=False)
print(f"  ✔  Calibrated passes saved: {len(df_passes)} rows")

## Section 9 — Collision Risk Scoring & ML Classifier

In [ ]:
# ============================================================
# SECTION 9 — COLLISION RISK SCORING & ML CLASSIFIER
# ============================================================

def score_collision_risk(df_feat: pd.DataFrame) -> pd.DataFrame:
    """
    Assign collision probability using a logistic model based on distance
    and relative speed. Also classifies into risk tiers:
      Low | Medium | High | Critical
    """
    if df_feat.empty:
        return df_feat

    # Logistic probability: closer + faster = higher risk
    d_norm = df_feat['distance_km'] / (THRESH_KM * 20)
    s_norm = df_feat['rel_speed']   / (df_feat['rel_speed'].max() + 1e-9)
    df_feat['collision_probability'] = (
        1 / (1 + np.exp(5 * d_norm - 3 * s_norm))
    ).clip(0, 1)

    def tier(p):
        if p < 0.001: return 'Low'
        if p < 0.005: return 'Medium'
        if p < 0.01:  return 'High'
        return 'Critical'

    df_feat['risk_tier'] = df_feat['collision_probability'].apply(tier)
    return df_feat


def train_collision_classifier(df_feat: pd.DataFrame) -> dict:
    """
    Train a Gradient Boosting Classifier to predict close-approach
    events (binary: label = 1 if distance < THRESH_KM).

    Features: distance_km, rel_speed, range_rate, hist_close_count
    Evaluation: ROC-AUC, Average Precision (PR-AUC)

    Returns dict with model, feature list, and metrics.
    """
    FEATURES = ['distance_km', 'rel_speed', 'range_rate', 'hist_close_count']
    feat_cols = [f for f in FEATURES if f in df_feat.columns]
    df_clean  = df_feat[feat_cols + ['label']].dropna()

    if df_clean['label'].sum() < 5:
        print("  ⚠  Too few positive labels to train classifier reliably.")
        return {}

    X = df_clean[feat_cols].values
    y = df_clean['label'].values
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42)

    clf = GradientBoostingClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, random_state=42, class_weight='balanced'
        if hasattr(GradientBoostingClassifier, 'class_weight') else None)
    clf.fit(X_tr, y_tr)

    y_prob = clf.predict_proba(X_te)[:, 1]
    roc    = roc_auc_score(y_te, y_prob) if len(np.unique(y_te)) > 1 else 0.0
    ap     = average_precision_score(y_te, y_prob)
    print(f"  Collision classifier — ROC-AUC: {roc:.3f} | PR-AUC: {ap:.3f}")

    # Save
    joblib.dump({'model': clf, 'features': feat_cols},
                os.path.join(WORKDIR, 'collision_model.pkl'))

    # ROC + PR curves
    plot_roc_pr(y_te, y_prob, roc, ap,
                save_path=os.path.join(EXPORT, 'roc_pr_curves.png'))

    return {'model': clf, 'features': feat_cols,
            'roc_auc': roc, 'pr_auc': ap}


def plot_roc_pr(y_test, y_prob, roc_auc, pr_auc, save_path=None):
    """Plot and save ROC curve and Precision-Recall curve."""
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    prec, rec, _ = precision_recall_curve(y_test, y_prob)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(fpr, tpr, color='steelblue', lw=2,
                 label=f'ROC-AUC = {roc_auc:.3f}')
    axes[0].plot([0, 1], [0, 1], '--', color='grey')
    axes[0].set(xlabel='FPR', ylabel='TPR', title='ROC Curve')
    axes[0].legend()

    axes[1].plot(rec, prec, color='darkorange', lw=2,
                 label=f'PR-AUC = {pr_auc:.3f}')
    axes[1].set(xlabel='Recall', ylabel='Precision',
                title='Precision-Recall Curve')
    axes[1].legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  ✔  Curves saved → {save_path}")
    plt.show()


df_features = score_collision_risk(df_features)
clf_result  = train_collision_classifier(df_features)
df_features.to_csv(os.path.join(EXPORT, 'collision_risk.csv'), index=False)

## Section 10 — Multi-Satellite Synthetic Dataset (100 Sats)

In [ ]:
# ============================================================
# SECTION 10 — MULTI-SATELLITE SYNTHETIC DATASET (100 SATS)
# ============================================================

def generate_synthetic_constellation(n_sats: int = N_SATS,
                                      n_frames: int = N_FRAMES,
                                      orbit_alt_km: float = ORBIT_ALT,
                                      noise_km: float = 2.5,
                                      forced_collisions: int = 40) -> pd.DataFrame:
    """
    Generate a realistic synthetic Starlink-like constellation with:
      - n_sats satellites in distributed orbital planes
      - n_frames propagation timesteps
      - Gaussian orbital noise (realistic position uncertainty)
      - Forced close-approach events for ML training

    Returns DataFrame with columns:
      [satellite_id, frame, timestamp, x_km, y_km, z_km,
       distance_km, rel_speed, range_rate, hist_close_count,
       prob_collision, brightness_mag]
    """
    ORBIT_R = R_EARTH + orbit_alt_km
    rng     = np.random.default_rng(42)

    # Distribute satellites across orbital planes
    n_planes = 8
    sats_per_plane = n_sats // n_planes
    inclinations = np.linspace(50, 70, n_planes)
    base_time = datetime(2025, 1, 1, tzinfo=timezone.utc)

    sat_params = []
    for p in range(n_planes):
        inc = np.radians(inclinations[p])
        raan = np.radians(p * 360 / n_planes)
        for s in range(sats_per_plane):
            phase = 2 * np.pi * s / sats_per_plane
            sat_params.append({
                'id': f'SAT_{p*sats_per_plane + s:03d}',
                'inc': inc, 'raan': raan, 'phase': phase
            })

    # Generate trajectories
    rows = []
    for t in range(n_frames):
        angle = 2 * np.pi * t / n_frames
        ts = base_time + timedelta(minutes=t * 5)
        for sp in sat_params:
            # Circular orbit with inclination and RAAN
            x = ORBIT_R * (
                np.cos(sp['raan']) * np.cos(sp['phase'] + angle)
                - np.sin(sp['raan']) * np.sin(sp['phase'] + angle) * np.cos(sp['inc']))
            y = ORBIT_R * (
                np.sin(sp['raan']) * np.cos(sp['phase'] + angle)
                + np.cos(sp['raan']) * np.sin(sp['phase'] + angle) * np.cos(sp['inc']))
            z = ORBIT_R * np.sin(sp['phase'] + angle) * np.sin(sp['inc'])
            # Add realistic orbital noise
            x += rng.normal(0, noise_km)
            y += rng.normal(0, noise_km)
            z += rng.normal(0, noise_km)
            rows.append({
                'satellite_id': sp['id'],
                'frame':        t,
                'timestamp':    ts,
                'x_km': x, 'y_km': y, 'z_km': z,
            })

    df = pd.DataFrame(rows)

    # Inject forced close-approach events
    sat_ids  = df['satellite_id'].unique()
    frame_ids = df['frame'].unique()
    for _ in range(forced_collisions):
        fi = rng.choice(frame_ids)
        sa, sb = rng.choice(sat_ids, 2, replace=False)
        mask_a = (df['satellite_id'] == sa) & (df['frame'] == fi)
        mask_b = (df['satellite_id'] == sb) & (df['frame'] == fi)
        if mask_a.sum() and mask_b.sum():
            ref = df.loc[mask_a, ['x_km', 'y_km', 'z_km']].iloc[0]
            df.loc[mask_b, 'x_km'] = ref.x_km + rng.uniform(0.5, 3.0)
            df.loc[mask_b, 'y_km'] = ref.y_km + rng.uniform(0.5, 3.0)
            df.loc[mask_b, 'z_km'] = ref.z_km + rng.uniform(0.5, 3.0)

    # Compute pairwise features per frame (vectorised where possible)
    print("  Computing pairwise distances (KD-Tree)...")
    result_rows = []
    hist_counts = {}

    for fi in tqdm(sorted(df['frame'].unique()), desc='Pairwise features'):
        snap = df[df['frame'] == fi]
        coords = snap[['x_km', 'y_km', 'z_km']].values
        ids    = snap['satellite_id'].values
        ts     = snap['timestamp'].iloc[0]

        if len(coords) < 2:
            continue

        # Velocity (finite diff) — zero for first frame
        if fi > 0:
            snap_prev = df[df['frame'] == fi - 1].set_index('satellite_id')
            vels = []
            for sid, row_i in zip(ids, coords):
                if sid in snap_prev.index:
                    prev = snap_prev.loc[sid, ['x_km', 'y_km', 'z_km']].values
                    vels.append((row_i - prev) / 300.0)  # 300s step
                else:
                    vels.append(np.zeros(3))
            vels = np.array(vels)
        else:
            vels = np.zeros_like(coords)

        tree  = cKDTree(coords)
        pairs = tree.query_pairs(r=SEARCH_R)

        for i, j in pairs:
            d   = np.linalg.norm(coords[i] - coords[j])
            dv  = vels[i] - vels[j]
            dr  = coords[i] - coords[j]
            rs  = float(np.linalg.norm(dv))
            rr  = float(np.dot(dv, dr) / (d + 1e-9))
            key = tuple(sorted((ids[i], ids[j])))
            hist_counts[key] = hist_counts.get(key, 0) + int(d < THRESH_KM)

            # Synthetic Pc: logistic on distance
            pc = float(1 / (1 + np.exp(0.5 * (d - THRESH_KM * 4))))
            # Brightness (average of two satellites)
            mag = simple_mag_model(elevation_deg=45.0)

            result_rows.append({
                'timestamp':         ts,
                'satellite_id':      ids[i],
                'frame':             fi,
                'x_km': coords[i,0], 'y_km': coords[i,1], 'z_km': coords[i,2],
                'distance_km':       d,
                'relative_speed':    rs,
                'range_rate':        rr,
                'hist_close_count':  hist_counts.get(key, 0),
                'collision_probability': pc,
                'brightness_mag':    mag,
                'sat_pair':          f"{ids[i]}__{ids[j]}",
            })

    df_out = pd.DataFrame(result_rows)
    print(f"  ✔  Synthetic dataset: {len(df_out):,} rows | "
          f"{df_out['satellite_id'].nunique()} satellites")
    return df_out


df_synth = generate_synthetic_constellation(
    n_sats=N_SATS, n_frames=N_FRAMES, forced_collisions=40)
df_synth.to_csv(os.path.join(EXPORT, 'multi_satellite_dataset.csv'), index=False)

## Section 11 — Plotly Interactive Dashboard (HTML Export)

In [ ]:
# ============================================================
# SECTION 11 — PLOTLY INTERACTIVE DASHBOARD (HTML EXPORT)
# ============================================================

def build_plotly_dashboard(df: pd.DataFrame,
                            save_path: str = None) -> go.Figure:
    """
    Build a multi-panel Plotly dashboard with:
      1. Top risky satellite pairs (bar chart, slider for threshold)
      2. 3D collision risk scatter (distance × speed × Pc)
      3. 3D orbital network snapshot (satellite positions)
      4. Pairwise risk heatmap (satA vs satB)
      5. Collision probability histogram
      6. Brightness distribution world map (if passes available)

    Returns a Plotly Figure.
    """
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]

    # Normalise column aliases
    rename = {
        'collision_probability': 'prob_collision',
        'relative_speed':        'rel_speed',
        'satellite_id':          'sat',
    }
    df.rename(columns={k: v for k, v in rename.items() if k in df.columns},
              inplace=True)

    # Required columns
    for col in ['prob_collision', 'rel_speed', 'distance_km', 'sat']:
        if col not in df.columns:
            df[col] = np.random.uniform(0, 0.01, len(df)) \
                      if col == 'prob_collision' else np.random.uniform(0, 100, len(df))

    # ── Panel 1: Top risky pairs ──────────────────────────
    top_pairs = (df.sort_values('prob_collision', ascending=False)
                   .drop_duplicates(subset=['sat'])
                   .head(20))

    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=[
            'Top 20 Satellites by Collision Probability',
            '3D Collision Risk (Distance × Speed × Pc)',
            '3D Orbital Network Snapshot',
            'Collision Probability Distribution',
            'Distance vs Collision Probability',
            'Brightness Distribution (Predicted Mag)',
        ],
        specs=[
            [{"type": "bar"},   {"type": "scatter3d"}],
            [{"type": "scatter3d"}, {"type": "histogram"}],
            [{"type": "scatter"},   {"type": "histogram"}],
        ],
        vertical_spacing=0.1, horizontal_spacing=0.08
    )

    # Bar chart
    fig.add_trace(go.Bar(
        x=top_pairs['sat'], y=top_pairs['prob_collision'],
        marker_color='steelblue', name='Collision Prob'), row=1, col=1)

    # 3D risk scatter
    sample = df.sample(min(2000, len(df)), random_state=42)
    fig.add_trace(go.Scatter3d(
        x=sample['distance_km'], y=sample['rel_speed'],
        z=sample['prob_collision'],
        mode='markers',
        marker=dict(size=2, color=sample['prob_collision'],
                    colorscale='Viridis', opacity=0.7),
        text=sample['sat'], name='Risk scatter'), row=1, col=2)

    # 3D orbital snapshot (first frame)
    if all(c in df.columns for c in ['x_km', 'y_km', 'z_km']):
        first_frame = df[df['frame'] == df['frame'].min()] if 'frame' in df.columns else df.head(200)
        fig.add_trace(go.Scatter3d(
            x=first_frame['x_km'], y=first_frame['y_km'], z=first_frame['z_km'],
            mode='markers',
            marker=dict(size=2, color=first_frame['prob_collision'],
                        colorscale='RdYlGn_r'),
            text=first_frame['sat'], name='Orbit snapshot'), row=2, col=1)

    # Histogram of Pc
    fig.add_trace(go.Histogram(
        x=df['prob_collision'], nbinsx=40,
        marker_color='salmon', name='Pc dist'), row=2, col=2)

    # Distance vs Pc scatter
    fig.add_trace(go.Scatter(
        x=sample['distance_km'], y=sample['prob_collision'],
        mode='markers', marker=dict(size=3, opacity=0.5, color='teal'),
        name='Dist vs Pc'), row=3, col=1)

    # Brightness histogram
    if 'brightness_mag' in df.columns:
        fig.add_trace(go.Histogram(
            x=df['brightness_mag'].dropna(), nbinsx=30,
            marker_color='goldenrod', name='Brightness'), row=3, col=2)
    elif df_passes is not None and 'calibrated_mag' in df_passes.columns:
        fig.add_trace(go.Histogram(
            x=df_passes['calibrated_mag'].dropna(), nbinsx=30,
            marker_color='goldenrod', name='Calibrated Mag'), row=3, col=2)

    fig.update_layout(
        title_text='Starlink Constellation Collision Risk Dashboard',
        height=1400, showlegend=False,
        paper_bgcolor='white', plot_bgcolor='#f9f9f9',
        font=dict(size=11)
    )

    if save_path:
        fig.write_html(save_path)
        print(f"  ✔  Dashboard saved → {save_path}")

    return fig


fig_dashboard = build_plotly_dashboard(
    df_synth,
    save_path=os.path.join(EXPORT, 'starlink_dashboard.html'))
fig_dashboard.show()

## Section 12 — Power BI Export Pipeline (6 Structured CSVs)

In [ ]:
# ============================================================
# SECTION 12 — POWER BI EXPORT PIPELINE (6 STRUCTURED CSVs)
# ============================================================

def export_powerbi_tables(df_synth: pd.DataFrame,
                           df_spacex_path: str = None,
                           out_dir: str = None) -> dict:
    """
    Generate 6 structured, Power BI-ready CSV tables from raw data.

    Tables:
      1. pbi_satellite_master   — dimension table (satellite attributes)
      2. pbi_collision_events   — fact table (encounter events)
      3. pbi_risk_summary       — per-satellite aggregated risk scores
      4. pbi_orbital_density    — KD-Tree density by altitude band
      5. pbi_launch_timeline    — cumulative launches over time
      6. pbi_kpi_summary        — single-row headline KPIs

    Separation of concerns: all computation here in Python,
    Power BI handles only visualization.
    """
    if out_dir is None:
        out_dir = EXPORT
    os.makedirs(out_dir, exist_ok=True)

    df = df_synth.copy()
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])

    # ── Table 2: Collision Events ─────────────────────────
    events = df[['timestamp', 'satellite_id', 'x_km', 'y_km', 'z_km',
                  'distance_km', 'relative_speed', 'range_rate',
                  'hist_close_count', 'collision_probability']].copy() \
             if 'satellite_id' in df.columns else df.copy()
    events.rename(columns={'satellite_id': 'satellite_id'}, inplace=True)

    def risk_tier(p):
        if pd.isna(p):    return 'Unknown'
        if p < 0.001:     return 'Low'
        elif p < 0.005:   return 'Medium'
        elif p < 0.01:    return 'High'
        return 'Critical'

    events['risk_tier']   = events['collision_probability'].apply(risk_tier)
    events['date']        = pd.to_datetime(events['timestamp']).dt.date
    events['hour']        = pd.to_datetime(events['timestamp']).dt.hour
    events['day_of_week'] = pd.to_datetime(events['timestamp']).dt.day_name()
    events['month']       = pd.to_datetime(events['timestamp']).dt.month_name()
    events.insert(0, 'event_id', range(1, len(events) + 1))
    events.to_csv(os.path.join(out_dir, 'pbi_collision_events.csv'), index=False)
    print(f"  pbi_collision_events.csv  — {len(events)} rows")

    # ── Table 3: Risk Summary ─────────────────────────────
    sat_col = 'satellite_id' if 'satellite_id' in events.columns else events.columns[0]
    risk_summary = events.groupby(sat_col).agg(
        total_events          = ('event_id',              'count'),
        avg_collision_prob    = ('collision_probability',  'mean'),
        max_collision_prob    = ('collision_probability',  'max'),
        min_distance_km       = ('distance_km',            'min'),
        avg_distance_km       = ('distance_km',            'mean'),
        avg_relative_speed    = ('relative_speed',         'mean'),
        total_hist_encounters = ('hist_close_count',       'sum'),
    ).reset_index()

    tier_counts = events.groupby([sat_col, 'risk_tier']).size().unstack(fill_value=0).reset_index()
    for t in ['Low', 'Medium', 'High', 'Critical']:
        if t not in tier_counts.columns:
            tier_counts[t] = 0
    risk_summary = risk_summary.merge(
        tier_counts[[sat_col, 'Low', 'Medium', 'High', 'Critical']],
        on=sat_col, how='left'
    )
    risk_summary.rename(columns={
        'Low': 'low_events', 'Medium': 'medium_events',
        'High': 'high_events', 'Critical': 'critical_events'
    }, inplace=True)

    max_prob = risk_summary['max_collision_prob'].max() or 1
    max_hist = risk_summary['total_hist_encounters'].max() or 1
    risk_summary['risk_score_composite'] = (
        0.5 * (risk_summary['avg_collision_prob'] / max_prob) * 100 +
        0.3 * (risk_summary['critical_events'] / (risk_summary['total_events'] + 1)) * 100 +
        0.2 * (risk_summary['total_hist_encounters'] / max_hist) * 100
    ).round(2)

    def overall_label(s):
        if s >= 60: return 'Critical'
        if s >= 35: return 'High'
        if s >= 15: return 'Medium'
        return 'Low'
    risk_summary['overall_risk_label'] = risk_summary['risk_score_composite'].apply(overall_label)
    risk_summary.to_csv(os.path.join(out_dir, 'pbi_risk_summary.csv'), index=False)
    print(f"  pbi_risk_summary.csv      — {len(risk_summary)} rows")

    # ── Table 4: Orbital Density (KD-Tree) ────────────────
    sat_positions = events.groupby(sat_col)[['x_km', 'y_km', 'z_km']].mean().dropna()
    coords   = sat_positions.values
    altitude = np.sqrt((coords**2).sum(axis=1)) - R_EARTH
    tree     = cKDTree(coords)
    n10  = [len(tree.query_ball_point(p, r=10))  - 1 for p in coords]
    n50  = [len(tree.query_ball_point(p, r=50))  - 1 for p in coords]
    n100 = [len(tree.query_ball_point(p, r=100)) - 1 for p in coords]

    density_df = pd.DataFrame({
        'satellite_id':    sat_positions.index,
        'x_km':            coords[:, 0],
        'y_km':            coords[:, 1],
        'z_km':            coords[:, 2],
        'altitude_km':     altitude,
        'neighbors_10km':  n10,
        'neighbors_50km':  n50,
        'neighbors_100km': n100,
    })
    bins = list(range(0, 2001, 50))
    density_df['altitude_band_km'] = pd.cut(
        density_df['altitude_km'], bins=bins,
        labels=[f'{b}-{b+50}' for b in bins[:-1]])
    density_df['density_risk'] = density_df['neighbors_10km'].apply(
        lambda n: 'Critical' if n >= 5 else ('High' if n >= 3 else ('Medium' if n >= 1 else 'Low')))
    density_df.to_csv(os.path.join(out_dir, 'pbi_orbital_density.csv'), index=False)
    print(f"  pbi_orbital_density.csv   — {len(density_df)} rows")

    # ── Table 1: Satellite Master ─────────────────────────
    master = density_df[['satellite_id', 'altitude_km', 'altitude_band_km']].copy()
    master['orbit_shell'] = master['altitude_km'].apply(lambda a:
        'VLEO (<400km)' if a < 400 else
        'Shell-1 (400-600km)' if a < 600 else
        'Shell-2 (600-800km)' if a < 800 else
        'Shell-3 (800-1200km)' if a < 1200 else 'MEO (>1200km)')
    master.to_csv(os.path.join(out_dir, 'pbi_satellite_master.csv'), index=False)
    print(f"  pbi_satellite_master.csv  — {len(master)} rows")

    # ── Table 5: Launch Timeline ──────────────────────────
    if 'timestamp' in events.columns:
        timeline = events.groupby(
            pd.Grouper(key='timestamp', freq='M')
        ).agg(launches_in_period=('satellite_id', 'nunique')).reset_index()
        timeline['launch_yearmonth'] = timeline['timestamp'].dt.to_period('M').astype(str)
        timeline['cumulative_launches'] = timeline['launches_in_period'].cumsum()
        timeline.to_csv(os.path.join(out_dir, 'pbi_launch_timeline.csv'), index=False)
        print(f"  pbi_launch_timeline.csv   — {len(timeline)} rows")
    else:
        timeline = pd.DataFrame()

    # ── Table 6: KPI Summary ──────────────────────────────
    kpi = pd.DataFrame([{
        'total_satellites':           master['satellite_id'].nunique(),
        'total_collision_events':     len(events),
        'critical_risk_events':       int((events['risk_tier'] == 'Critical').sum()),
        'high_risk_events':           int((events['risk_tier'] == 'High').sum()),
        'avg_collision_probability':  round(float(events['collision_probability'].mean()), 6),
        'max_collision_probability':  round(float(events['collision_probability'].max()), 6),
        'min_observed_distance_km':   round(float(events['distance_km'].min()), 4),
        'avg_observed_distance_km':   round(float(events['distance_km'].mean()), 4),
        'kdtree_close_pairs_10km':    sum(n10) // 2,
        'avg_neighbors_within_50km':  round(float(np.mean(n50)), 2),
        'export_timestamp':           datetime.utcnow().isoformat(),
    }])
    kpi.to_csv(os.path.join(out_dir, 'pbi_kpi_summary.csv'), index=False)
    print(f"  pbi_kpi_summary.csv       — 1 row")

    print(f"\n  ✔  All Power BI tables exported to: {out_dir}")
    return {
        'events': events, 'risk_summary': risk_summary,
        'density': density_df, 'master': master,
        'timeline': timeline, 'kpi': kpi,
    }


pbi_tables = export_powerbi_tables(df_synth, out_dir=EXPORT)

## Section 13 — Summary & File Export

In [ ]:
# ============================================================
# SECTION 13 — SUMMARY & FILE EXPORT
# ============================================================

def print_summary(pbi_tables: dict, clf_result: dict, mobj: dict):
    """Print a structured run summary."""
    kpi = pbi_tables['kpi'].iloc[0]
    print("\n" + "=" * 60)
    print("  STARLINK COLLISION RISK ANALYSIS — RUN SUMMARY")
    print("=" * 60)
    print(f"  Total satellites analyzed  : {kpi['total_satellites']}")
    print(f"  Total collision events     : {kpi['total_collision_events']:,}")
    print(f"  Critical risk events       : {kpi['critical_risk_events']}")
    print(f"  High risk events           : {kpi['high_risk_events']}")
    print(f"  Avg collision probability  : {kpi['avg_collision_probability']:.6f}")
    print(f"  Min observed distance (km) : {kpi['min_observed_distance_km']:.4f}")
    print(f"  KD-Tree close pairs <10km  : {kpi['kdtree_close_pairs_10km']}")
    print(f"  Avg neighbors within 50km  : {kpi['avg_neighbors_within_50km']}")
    if clf_result:
        print(f"  Collision classifier ROC   : {clf_result.get('roc_auc', 0):.3f}")
        print(f"  Collision classifier PR    : {clf_result.get('pr_auc', 0):.3f}")
    print("=" * 60)

    # Save JSON summary
    summary = {
        'run_timestamp':        datetime.utcnow().isoformat(),
        'total_satellites':     int(kpi['total_satellites']),
        'total_events':         int(kpi['total_collision_events']),
        'critical_events':      int(kpi['critical_risk_events']),
        'avg_pc':               float(kpi['avg_collision_probability']),
        'min_distance_km':      float(kpi['min_observed_distance_km']),
        'kdtree_pairs_10km':    int(kpi['kdtree_close_pairs_10km']),
        'clf_roc_auc':          clf_result.get('roc_auc', None),
        'clf_pr_auc':           clf_result.get('pr_auc', None),
        'outputs': [
            'collision_features.csv',
            'collision_risk.csv',
            'passes_calibrated.csv',
            'calibration_data.csv',
            'multi_satellite_dataset.csv',
            'pbi_satellite_master.csv',
            'pbi_collision_events.csv',
            'pbi_risk_summary.csv',
            'pbi_orbital_density.csv',
            'pbi_launch_timeline.csv',
            'pbi_kpi_summary.csv',
            'starlink_dashboard.html',
            'altitude_density.png',
            'roc_pr_curves.png',
        ]
    }
    with open(os.path.join(WORKDIR, 'run_summary.json'), 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"\n  ✔  Summary saved → {os.path.join(WORKDIR, 'run_summary.json')}")
    print(f"  ✔  All outputs in: {EXPORT}")


print_summary(pbi_tables, clf_result, mobj)

# ── Zip and download in Colab ──────────────────────────────
import shutil
zip_path = os.path.join(WORKDIR, 'starlink_export_all')
shutil.make_archive(zip_path, 'zip', EXPORT)
print(f"\n  ✔  ZIP ready: {zip_path}.zip")

# Uncomment in Colab to download:
# from google.colab import files
# files.download(f'{zip_path}.zip')